In [140]:
from rich.console import Console
from openai import OpenAI
from dotenv import load_dotenv
import os
import json
load_dotenv(override=True)
import requests
import gradio as gr



In [122]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [123]:
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
groq_api_key = os.getenv('GROQ_API_KEY')
groq = OpenAI(base_url=GROQ_BASE_URL,api_key=groq_api_key)

In [124]:
checklist = []
completed = []

In [125]:
def get_checklist_report() -> str:
    result = ''
    for index,item in enumerate(checklist):
        if completed[index]:
            result += f"Checklist #{index+1} : [green][strike]{item}[/strike][/green]\n"
        else:
            result += f"Checklist #{index+1} : {item}\n"
    show(result)
    return result

In [126]:
def create_checklist(descriptions: list[str]) -> str:
    checklist.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_checklist_report()

In [127]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(checklist):
        completed[index - 1] = True
    else:
        return "No checlist completed"
    Console().print(completion_notes)
    return get_checklist_report()

In [128]:
def check_streaming_availability(movie_title: str, country: str = "IN") -> str:
    api_key = os.getenv('TMDB_API_KEY')
    
    search = requests.get(
        f"https://api.themoviedb.org/3/search/movie?query={movie_title}&api_key={api_key}"
    ).json()
    
    if not search.get("results"):
        return json.dumps({"error": f"Movie '{movie_title}' not found"})
    
    movie_id = search["results"][0]["id"]
    movie_name = search["results"][0]["title"]
    
    providers = requests.get(
        f"https://api.themoviedb.org/3/movie/{movie_id}/watch/providers?api_key={api_key}"
    ).json()
    
    country_data = providers.get("results", {}).get(country, {})
    
    available_on = []
    for provider_type in ["flatrate", "rent", "buy"]:
        for p in country_data.get(provider_type, []):
            available_on.append({"name": p["provider_name"], "type": provider_type})
    
    if not available_on:
        return json.dumps({"movie": movie_name, "streaming": "Not available for streaming in " + country})
    
    return json.dumps({"movie": movie_name, "country": country, "available_on": available_on})


In [129]:
create_checklist_json = {
    'name':'create_checklist',
    'description':'Use this tool to add new checklist from a list of descriptions and return the full list',
    'parameters':{
        'type':'object',
        'properties':{
            'descriptions':{
                'type':'array',
                'items':{'type':'string'},
                'title': 'Descriptions of checklist items'
            }
        },
        'required':['descriptions'],
        'additionalProperties':False
    }
}

In [130]:
mark_complete_json = {
    'name':'mark_complete',
    'description':'Mark complete the checklist item at the given position (starting from 1) and return the full list',
    'parameters':{
        'properties':{
            'index':{
                'description': 'The 1-based index of the checklist item to mark as complete',
                'title': 'Index',
                'type': 'integer'
            },
            'completion_notes':{
                'description': 'Notes about how you completed the checklist item in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
            }
        },
        'required':['index','completion_notes'],
        'type':'object',
        'additionalProperties':False
    }
}

In [131]:
check_streaming_json = {
    'name': 'check_streaming_availability',
    'description': 'Check which streaming platforms (Netflix, Prime, etc.) a movie is currently available on in a given country. Use this to tell the user where they can watch the recommended movie.',
    'parameters': {
        'type': 'object',
        'properties': {
            'movie_title': {
                'type': 'string',
                'description': 'The title of the movie to check'
            },
            'country': {
                'type': 'string',
                'description': 'ISO 3166-1 country code (e.g. IN for India, US for United States)',
                'default': 'IN'
            }
        },
        'required': ['movie_title'],
        'additionalProperties': False
    }
}


In [132]:
tools = [
    {"type": "function", "function": create_checklist_json},
    {"type": "function", "function": mark_complete_json},
    {"type": "function", "function": check_streaming_json}
]


In [133]:
tools

[{'type': 'function',
  'function': {'name': 'create_checklist',
   'description': 'Use this tool to add new checklist from a list of descriptions and return the full list',
   'parameters': {'type': 'object',
    'properties': {'descriptions': {'type': 'array',
      'items': {'type': 'string'},
      'title': 'Descriptions of checklist items'}},
    'required': ['descriptions'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'mark_complete',
   'description': 'Mark complete the checklist item at the given position (starting from 1) and return the full list',
   'parameters': {'properties': {'index': {'description': 'The 1-based index of the checklist item to mark as complete',
      'title': 'Index',
      'type': 'integer'},
     'completion_notes': {'description': 'Notes about how you completed the checklist item in rich console markup',
      'title': 'Completion Notes',
      'type': 'string'}},
    'required': ['index', 'completion_notes'],
  

In [134]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role":"tool","content":json.dumps(result),"tool_call_id":tool_call.id})
    return results

In [135]:
def loop(messages):
    response = groq.chat.completions.create(model='openai/gpt-oss-120b',messages=messages,tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        result = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(result)
        response = groq.chat.completions.create(model='openai/gpt-oss-120b',messages=messages,tools=tools)
    show(response.choices[0].message.content)

In [136]:
system_message = """
You are a Movie Night Decider agent. The user gives you a mood, genre, or vibe and you help them pick the perfect movie.

Use your checklist tools to plan and execute your recommendation process:
1. First, create a checklist of steps to narrow down the perfect movie.
2. Then work through each step, marking it complete with detailed notes.
3. After selecting your top pick, use check_streaming_availability to find where the user can actually watch it right now.

Your steps should include: identifying the mood/genre, brainstorming 5 candidate movies, filtering by runtime (under 2.5 hours), ranking by quality/popularity, presenting the final pick with a short pitch, and checking streaming availability.

If the user is vague, make reasonable assumptions. Do not ask the user any follow-up questions.
Provide your final recommendation in Rich console markup without code blocks.
"""


In [137]:
user_message = "I'm in the mood for something mind-bending and thrilling — a movie that keeps me guessing till the very end."


In [138]:
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [139]:
checklist, completed = [], []
loop(messages)

Checklist #1 : Identify the mood/genre: mind-bending, thrilling, keeps guessing till the end.
Checklist #2 : Brainstorm 5 candidate movies that fit the mood/genre.
Checklist #3 : Filter the candidate movies by runtime (under 150 minutes).
Checklist #4 : Rank the filtered movies by quality and popularity.
Checklist #5 : Select the top pick and craft a short pitch.
Checklist #6 : Check streaming availability for the top pick.

User wants a mind‑bending, thriller that keeps them guessing. This points to high‑concept, plot‑twist‑heavy films 
often in the sci‑fi, psychological thriller, or neo‑noir categories.

Checklist #1 : Identify the mood/genre: mind-bending, thrilling, keeps guessing till the end.
Checklist #2 : Brainstorm 5 candidate movies that fit the mood/genre.
Checklist #3 : Filter the candidate movies by runtime (under 150 minutes).
Checklist #4 : Rank the filtered movies by quality and popularity.
Checklist #5 : Select the top pick and craft a short pitch.
Checklist #6 : Check streaming availability for the top pick.

Brainstormed five movies that are renowned for mind‑bending plots and thrilling twists:
1. **Inception** (2010) – 148 min
2. **Shutter Island** (2010) – 138 min
3. **The Prestige** (2006) – 130 min
4. **Memento** (2000) – 113 min
5. **Gone Girl** (2014) – 149 min

Checklist #1 : Identify the mood/genre: mind-bending, thrilling, keeps guessing till the end.
Checklist #2 : Brainstorm 5 candidate movies that fit the mood/genre.
Checklist #3 : Filter the candidate movies by runtime (under 150 minutes).
Checklist #4 : Rank the filtered movies by quality and popularity.
Checklist #5 : Select the top pick and craft a short pitch.
Checklist #6 : Check streaming availability for the top pick.

...

**Top Pick: Inception (2010)**

*Why it fits:* A masterclass in mind‑bending storytelling, blending heist thrills with layered dream‑world puzzles.
Every scene pulls you deeper, and the ending leaves you questioning reality—exactly the guessing‑till‑the‑last 
experience you crave.

*Short Pitch:* A skilled thief (Leonardo DiCaprio) enters people’s dreams to steal secrets, but his biggest job is 
to plant an idea—an "inception." As the team navigates collapsing dream‑levels, the line between what's real and 
imagined blurs, culminating in a heart‑pounding finale that will have you debating the spinning top long after the 
credits roll.

Checklist #1 : Identify the mood/genre: mind-bending, thrilling, keeps guessing till the end.
Checklist #2 : Brainstorm 5 candidate movies that fit the mood/genre.
Checklist #3 : Filter the candidate movies by runtime (under 150 minutes).
Checklist #4 : Rank the filtered movies by quality and popularity.
Checklist #5 : Select the top pick and craft a short pitch.
Checklist #6 : Check streaming availability for the top pick.

In India (IN), *Inception* is currently available to stream on **Amazon Prime Video** (flatrate) and **JioSaavn** 
(flatrate). It can also be rented or purchased on Apple TV Store, Google Play Movies, YouTube, and Amazon Video.

Checklist #1 : Identify the mood/genre: mind-bending, thrilling, keeps guessing till the end.
Checklist #2 : Brainstorm 5 candidate movies that fit the mood/genre.
Checklist #3 : Filter the candidate movies by runtime (under 150 minutes).
Checklist #4 : Rank the filtered movies by quality and popularity.
Checklist #5 : Select the top pick and craft a short pitch.
Checklist #6 : Check streaming availability for the top pick.

**🎬 Movie Night Recommendation**

**Top Pick:** *Inception* (2010) – 148 min  

**Why it fits:** A masterclass in mind‑bending storytelling, blending heist thrills with layered dream‑world 
puzzles. Every scene pulls you deeper, and the ending leaves you questioning reality—exactly the 
guessing‑till‑the‑last experience you crave.

**Short Pitch:**  
A skilled thief (Leonardo DiCaprio) enters people’s dreams to steal secrets, but his biggest job is to plant an 
idea—an “inception.” As the team navigates collapsing dream‑levels, the line between what's real and imagined 
blurs, culminating in a heart‑pounding finale that will have you debating the spinning top long after the credits 
roll.

**Where to watch (India):**  
- **Amazon Prime Video** (flatrate)  
- **JioSaavn** (flatrate)  

*Also available to rent or buy on Apple TV Store, Google Play Movies, YouTube, and Amazon Video.* Enjoy the 
mind‑bending ride!